# Where Python Actually Starts

This notebook has been rewritten to be slower, more reflective, and less template-like. Instead of treating **Where Python Actually Starts** as a small isolated topic, the goal here is to treat it as part of Python's larger runtime story: objects are created, names are bound, frames execute bytecode, and the interpreter keeps following rules whether or not the source code looks simple from the outside.

If this topic ever felt a little too neat or too magical when explained quickly, that is exactly what this rewrite is trying to fix. We are going to keep asking two grounding questions: what changed in memory, and what instructions is the interpreter actually stepping through under the surface?

The point is not to drown in internals. The point is to make the language feel explainable. Once that happens, even advanced behavior starts feeling much less mysterious.


## Why this topic becomes easier once you watch the runtime carefully

When people struggle with **Where Python Actually Starts**, the struggle is often not about syntax. It is usually about trying to reason from surface appearance alone. Python rewards a deeper habit: do not ask only what the code *looks like*; ask what objects exist, what names or attributes point to them, what bytecode-level actions are likely happening, and which runtime protocol is being used.

That habit is what turns memorized rules into real understanding. It also makes interviews easier, because you stop giving slogan answers and start giving mechanism-based answers.


## Questions worth answering before we go any further

- See the difference between source code, bytecode, and runtime execution
- Understand what a frame is in practical terms
- Separate names from objects early
- Build a mental picture that will support everything else


## What changes in memory while this code runs

At runtime, Python keeps namespaces that map names to objects. A frame carries local names, links to global names, and information about where execution currently is. The important human insight is this: values do not float around alone. They are reached through names, containers, attributes, and call frames.


In [1]:
x = 42
text = "python"
values = [x, text]
print("id(x):", id(x))
print("id(text):", id(text))
print("id(values):", id(values))
print(values)


id(x): 140715746449480
id(text): 2170576422448
id(values): 2170627927040
[42, 'python']


## What compiled bytecode reveals about this code

Bytecode is not something you must memorize, but it is the best way to see that Python lowers your code into smaller runtime instructions. Seeing it once makes the interpreter feel less mystical and more concrete.


In [2]:
import dis

def total(n):
    s = 0
    for i in range(n):
        s += i
    return s

dis.dis(total)


  3           0 RESUME                   0

  4           2 LOAD_CONST               1 (0)
              4 STORE_FAST               1 (s)

  5           6 LOAD_GLOBAL              1 (NULL + range)
             18 LOAD_FAST                0 (n)
             20 PRECALL                  1
             24 CALL                     1
             34 GET_ITER
        >>   36 FOR_ITER                 7 (to 52)
             38 STORE_FAST               2 (i)

  6          40 LOAD_FAST                1 (s)
             42 LOAD_FAST                2 (i)
             44 BINARY_OP               13 (+=)
             48 STORE_FAST               1 (s)
             50 JUMP_BACKWARD            8 (to 36)

  7     >>   52 LOAD_FAST                1 (s)
             54 RETURN_VALUE


## A slower walk through the main ideas

### Source text is only the beginning
The file or notebook cell you write is only the human-friendly layer. Python still has to parse it and turn it into executable instructions.

### Execution happens inside frames
Whenever you call a function, Python creates a new execution frame. That frame holds local variables and remembers where control should return afterward.

### Names are entries in namespaces
The name `x` is not the integer itself. It is a reference in some namespace that points to an object.

### A runtime model beats memorized rules
Many “advanced” Python facts stop feeling advanced once you keep the frame and object model in your head.


## Watching function calls create separate working spaces

Each call to `describe` gets its own local variables, even though the function body is the same.


In [3]:
def describe(value):
    local_message = f"inside with {value!r}"
    print(local_message)

describe(10)
describe("python")


inside with 10
inside with 'python'


## A stack trace is a map of frames

This exception is intentional. The traceback is showing the chain of frames that led to the problem.


In [4]:
def a():
    b()

def b():
    c()

def c():
    raise ValueError("trace me")

try:
    a()
except ValueError as exc:
    import traceback
    traceback.print_exc()


Traceback (most recent call last):
  File "C:\Users\VIHAAN\AppData\Local\Temp\ipykernel_25316\2221454973.py", line 11, in <module>
    a()
  File "C:\Users\VIHAAN\AppData\Local\Temp\ipykernel_25316\2221454973.py", line 2, in a
    b()
  File "C:\Users\VIHAAN\AppData\Local\Temp\ipykernel_25316\2221454973.py", line 5, in b
    c()
  File "C:\Users\VIHAAN\AppData\Local\Temp\ipykernel_25316\2221454973.py", line 8, in c
    raise ValueError("trace me")
ValueError: trace me


## Code objects exist even before a function runs

A function carries a code object that stores compiled information about the function body.


In [5]:
def sample(a, b):
    return a + b

print(sample.__code__)
print(sample.__code__.co_varnames)


<code object sample at 0x000001F96382C5E0, file "C:\Users\VIHAAN\AppData\Local\Temp\ipykernel_25316\2060303189.py", line 1>
('a', 'b')


## Where this usually shows up in real code

This is not only a classroom topic. **Where Python Actually Starts** usually appears in real projects as a debugging problem, a design choice, a performance tradeoff, or a readability decision. In other words, this topic matters most when the codebase gets large enough that vague intuition stops being reliable.

That is why the low-level view is useful. It gives you something firmer than guesswork when code starts behaving in surprising ways.


## Common traps that still catch careful people

- Thinking a variable is the same thing as the object it names
- Treating bytecode as irrelevant trivia instead of a useful mental tool
- Ignoring frames when reading tracebacks


## Questions that reveal whether this idea is really clear yet

- What happens between Python source code and execution?
- What is a frame?
- What is the difference between a name and an object?


## What I would really want you to remember later

- Python executes compiled instructions, not raw text.
- Frames are the working spaces of execution.
- Names point to objects through namespaces.


## One last grounding thought

If this notebook did its job, **Where Python Actually Starts** should now feel less like a bag of special rules and more like a natural consequence of Python's runtime model. That is the standard we want: not just recall, but a calm explanation of what the interpreter is seeing and why the behavior follows from that.
